**1: Data Ingestion and Environment Setup**

This notebook serves as the foundation for our research comparing XLM-R and AfriBERTa for sentiment analysis in African languages. Our primary goal here is to establish a robust directory structure and acquire the necessary datasets for our target languages: Hausa and Kinyarwanda. By the end of this notebook, we will have a cleaned, standardized dataset stored in Google Drive, ready for model fine-tuning and subsequent Explainable AI (XAI) analysis.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Define your root project path
PROJECT_PATH = '/content/drive/Shareddrives/Cos760'

# List of subfolders to create
folders = [
    'data/raw',
    'data/processed',
    'models/xlmr',
    'models/afriberta',
    'outputs/xai',
    'outputs/metrics'
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print("Project structure created successfully!")

Project structure created successfully!


**2: Data Ingestion from AfriSenti Hub**

In this step, we use the Hugging Face datasets library to pull the AfriSenti benchmark dataset. AfriSenti is a multi-language Twitter sentiment dataset specifically designed for African languages. We are focusing on Hausa (hau) and Kinyarwanda (kin) to represent different linguistic families.

We save these datasets directly to disk in the Arrow format within our data/raw directory. This approach is more efficient than re-downloading the data in every session and ensures we are working with a consistent version of the data throughout the project lifecycle.

In [3]:
!pip install datasets

from datasets import load_dataset
import os

PROJECT_PATH = '/content/drive/Shareddrives/Cos760'
# This path supports the newer 'no-script' security policy
DATASET_HUB_PATH = "masakhane/afrisenti"

languages = {'hausa': 'hau', 'kinyarwanda': 'kin'}

for lang_name, lang_code in languages.items():
    print(f"--- Processing {lang_name} ({lang_code}) ---")

    try:
        # trust_remote_code=True is sometimes still needed for specific legacy metadata,
        # but the HausaNLP path usually bypasses the script error entirely.
        dataset = load_dataset(DATASET_HUB_PATH, lang_code)

        save_path = os.path.join(PROJECT_PATH, 'data/raw', lang_name)
        os.makedirs(save_path, exist_ok=True)

        dataset.save_to_disk(save_path)
        print(f"✅ Successfully saved {lang_name} to {save_path}")

    except Exception as e:
        print(f"❌ Failed to load {lang_name}: {e}")

print("\nFinal check of data/raw directory:")
print(os.listdir(os.path.join(PROJECT_PATH, 'data/raw')))

--- Processing hausa (hau) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.tsv: 0.00B [00:00, ?B/s]

dev.tsv: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/1026 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3302 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/827 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1026 [00:00<?, ? examples/s]

✅ Successfully saved kinyarwanda to /content/drive/Shareddrives/Cos760/data/raw/kinyarwanda

Final check of data/raw directory:
['hausa', 'kinyarwanda']


**3: Data Loading and Inspection**

Now that the raw datasets are saved to disk in Arrow format, we need to load them into Pandas DataFrames. This allows us to easily clean the text and perform Exploratory Data Analysis (EDA) to understand the class balance for Hausa and Kinyarwanda.

In [4]:
import pandas as pd
from datasets import load_from_disk

# Dictionary to store our dataframes
dfs = {}

for lang_name in languages.keys():
    load_path = os.path.join(PROJECT_PATH, 'data/raw', lang_name)
    dataset = load_from_disk(load_path)

    # Convert splits to pandas for easier manipulation
    dfs[lang_name] = {
        'train': dataset['train'].to_pandas(),
        'validation': dataset['validation'].to_pandas(),
        'test': dataset['test'].to_pandas()
    }

    print(f"✅ Loaded {lang_name} splits: "
          f"Train ({len(dfs[lang_name]['train'])}), "
          f"Val ({len(dfs[lang_name]['validation'])}), "
          f"Test ({len(dfs[lang_name]['test'])})")

# Preview Hausa train data
print("\nPreview of Hausa Train Data:")
display(dfs['hausa']['train'].head())

✅ Loaded hausa splits: Train (14172), Val (2677), Test (5303)
✅ Loaded kinyarwanda splits: Train (3302), Val (827), Test (1026)

Preview of Hausa Train Data:


,tweet,label
0,@user Da kudin da Arewa babu wani abin azo aga...,negative
1,@user Kaga wani Adu ar Banda💔😭 wai a haka Shi ...,negative
2,@user Sai haquri fa yan madrid daman kunce cha...,negative
3,@user Hmmm yanzu kai kasan girman allah daxaka...,negative
4,@user @user Wai gwamno nin Nigeria suna afa kw...,negative


**4: Exploratory Data Analysis (EDA)**

Before training, we must check if the datasets are balanced. If one language has significantly more "positive" tweets than "negative" ones, our XLM-R and AfriBERTa models might develop a bias.

In [5]:
import re

def clean_tweet(text):
    # Remove @user handles
    text = re.sub(r'@\w+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the cleaning function to all languages and all splits
for lang in dfs:
    for split in ['train', 'validation', 'test']:
        print(f"Cleaning {lang} - {split}...")
        dfs[lang][split]['cleaned_tweet'] = dfs[lang][split]['tweet'].apply(clean_tweet)

# Verify the cleaning
if 'hausa' in dfs:
    print("\nComparison for Hausa:")
    display(dfs['hausa']['train'][['tweet', 'cleaned_tweet']].head())

Cleaning hausa - train...
Cleaning hausa - validation...
Cleaning hausa - test...
Cleaning kinyarwanda - train...
Cleaning kinyarwanda - validation...
Cleaning kinyarwanda - test...

Comparison for Hausa:


,tweet,cleaned_tweet
0,@user Da kudin da Arewa babu wani abin azo aga...,Da kudin da Arewa babu wani abin azo agani da ...
1,@user Kaga wani Adu ar Banda💔😭 wai a haka Shi ...,Kaga wani Adu ar Banda💔😭 wai a haka Shi ne shu...
2,@user Sai haquri fa yan madrid daman kunce cha...,Sai haquri fa yan madrid daman kunce champion ...
3,@user Hmmm yanzu kai kasan girman allah daxaka...,Hmmm yanzu kai kasan girman allah daxakace muk...
4,@user @user Wai gwamno nin Nigeria suna afa kw...,Wai gwamno nin Nigeria suna afa kwayoyi ko 😂


**5: Final Export of Processed Data**

The final step of this notebook is to save our cleaned data into the data/processed directory. We save them as CSV files so that our next notebook (02_Model_Training) can load the clean data directly without needing to repeat the ingestion logic.

In [ ]:
# Export cleaned dataframes to CSV
for lang in dfs:
    
        out_filename = f"{lang}_{split}_cleaned.csv"
        out_path = os.path.join(PROJECT_PATH, 'data/processed', out_filename)

        # We only save the text and the label to keep the file light
        export_df = dfs[lang][split][['cleaned_tweet', 'label']]
        export_df.to_csv(out_path, index=False)

        print(f"💾 Saved: {out_path}")

print("\n--- Notebook 01 Finished ---")
print("Check your 'data/processed' folder for the CSV files.")

💾 Saved: /content/drive/Shareddrives/Cos760/data/processed/hausa_train_cleaned.csv
💾 Saved: /content/drive/Shareddrives/Cos760/data/processed/hausa_validation_cleaned.csv
💾 Saved: /content/drive/Shareddrives/Cos760/data/processed/hausa_test_cleaned.csv
💾 Saved: /content/drive/Shareddrives/Cos760/data/processed/kinyarwanda_train_cleaned.csv
💾 Saved: /content/drive/Shareddrives/Cos760/data/processed/kinyarwanda_validation_cleaned.csv
💾 Saved: /content/drive/Shareddrives/Cos760/data/processed/kinyarwanda_test_cleaned.csv

--- Notebook 01 Finished ---
Check your 'data/processed' folder for the CSV files.
